# Extract Residual Stream Activations

This notebook demonstrates how to capture residual-stream activations during
vLLM inference using the async engine and vllm-lens.

## Imports

In [1]:
from vllm import AsyncEngineArgs, AsyncLLMEngine, SamplingParams

## Configuration

In [2]:
MODEL = "facebook/opt-125m"
PROMPT = "The future of AI is"
LAYERS = [0, 2, 5]

## Initialize the Async Engine

In [3]:
engine_args = AsyncEngineArgs(
    model=MODEL,
    gpu_memory_utilization=0.85,
)
engine = AsyncLLMEngine.from_engine_args(engine_args)

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

INFO 03-14 09:58:52 [model.py:531] Resolved architecture: OPTForCausalLM
INFO 03-14 09:58:52 [model.py:1554] Using max model len 2048
INFO 03-14 09:58:52 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 03-14 09:58:52 [vllm.py:747] Asynchronous scheduling is enabled.
WARNING 03-14 09:58:52 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 03-14 09:58:52 [vllm.py:792] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 03-14 09:58:52 [vllm.py:957] Cudagraph is disabled under eager mode


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

(EngineCore_DP0 pid=278813) INFO 03-14 09:58:55 [core.py:101] Initializing a V1 LLM engine (v0.17.0) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False,

Loading pt checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore_DP0 pid=278813) INFO 03-14 09:59:04 [default_loader.py:293] Loading weights took 0.14 seconds
(EngineCore_DP0 pid=278813) INFO 03-14 09:59:04 [gpu_model_runner.py:4338] Model loading took 0.24 GiB memory and 5.151460 seconds
(EngineCore_DP0 pid=278813) INFO 03-14 09:59:09 [gpu_worker.py:424] Available KV cache memory: 80.23 GiB
(EngineCore_DP0 pid=278813) INFO 03-14 09:59:09 [kv_cache_utils.py:1314] GPU KV cache size: 2,336,944 tokens
(EngineCore_DP0 pid=278813) INFO 03-14 09:59:09 [kv_cache_utils.py:1319] Maximum concurrency for 2,048 tokens per request: 1141.09x
(EngineCore_DP0 pid=278813) INFO 03-14 09:59:10 [core.py:282] init engine (profile, create kv cache, warmup model) took 5.54 seconds
(EngineCore_DP0 pid=278813) INFO 03-14 09:59:10 [vllm.py:957] Cudagraph is disabled under eager mode
WARNING 03-14 09:59:10 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore_DP0 pid

## Generate with Activation Capture

Pass `output_residual_stream` via `extra_args` to request residual-stream
activations at the specified layers.

In [4]:
sampling_params = SamplingParams(
    temperature=0.7,
    max_tokens=64,
    extra_args={"output_residual_stream": LAYERS},
)

final = None
async for output in engine.generate(PROMPT, sampling_params, "req-0"):
    final = output

WARNING 03-14 09:59:10 [input_processor.py:254] Passing raw prompts to InputProcessor is deprecated and will be removed in v0.18. You should instead pass the outputs of Renderer.render_cmpl() or Renderer.render_chat().


## Inspect Results

In [5]:
print(f"Prompt: {PROMPT}")
print(f"Generated: {final.outputs[0].text}")

activation_data = getattr(final, "activations", None)
if activation_data is not None:
    residual_stream = activation_data["residual_stream"]
    print(f"\nResidual stream shape: {residual_stream.shape}")
    print(f"Sample activations (layer 0, first token): {residual_stream[0, 0, :5]}")
else:
    print("Activations: NOT FOUND (ERROR)")

Prompt: The future of AI is
Generated:  cloud computing

Jose Antonia

AI is destined to become a future of computing. The future of computing will be cloud computing, where it is no longer an urban fantasy.

The future of computing will be cloud computing, where it is no longer an urban fantasy. The future of computing will be cloud

Residual stream shape: torch.Size([3, 69, 768])
Sample activations (layer 0, first token): tensor([-0.1010, -1.0039,  0.1498,  0.1503,  0.0785], dtype=torch.float16)
